# 06 — Model Evaluation & Comparison

Load all 4 trained model checkpoints and evaluate on the shared held-out test set.  
Generates: confusion matrices, ROC curves, PR curves, per-class F1 bar charts,
inference speed benchmark, and the full summary table.

> **Prerequisites:** Run notebooks 02–05 first to generate checkpoints in `results/baseline/`

In [ ]:
import os, sys, time
import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sys.path.insert(0, os.path.abspath('..'))
from src.dataset import get_dataloaders
from src.models import MODEL_REGISTRY
from src.evaluate import (
    evaluate_model, plot_per_class_f1,
    get_predictions, compute_metrics,
    plot_confusion_matrix, plot_roc_curves, plot_pr_curves,
    CLASS_NAMES, SHORT_NAMES
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
X = np.load('../data/X.npy')
y = np.load('../data/y.npy')
_, _, test_loader, _ = get_dataloaders(X, y, batch_size=256)
print(f'Test samples: {len(test_loader.dataset)}')

In [ ]:
MODEL_NAMES = ['cnn', 'resnet', 'lstm', 'transformer']
DISPLAY = {'cnn':'1D-CNN', 'resnet':'ResNet-1D', 'lstm':'BiLSTM', 'transformer':'CNN-Transformer'}
BASELINE_DIR = '../results/baseline'

all_metrics, all_preds = {}, {}

for name in MODEL_NAMES:
    ckpt_path = f'{BASELINE_DIR}/{name}_best.pth'
    if not os.path.exists(ckpt_path):
        print(f'[skip] {name} checkpoint not found')
        continue
    ckpt = torch.load(ckpt_path, map_location=device)
    model = MODEL_REGISTRY[name](n_leads=2, n_classes=5).to(device)
    model.load_state_dict(ckpt['model_state'])

    metrics, y_true, y_pred, y_prob = evaluate_model(
        model, test_loader, device,
        save_dir=BASELINE_DIR, model_name=name
    )

    # inference speed
    model.eval()
    times = []
    with torch.no_grad():
        for i, (Xb, _) in enumerate(test_loader):
            if i >= 10: break
            Xb = Xb.to(device)
            t0 = time.perf_counter()
            _ = model(Xb)
            times.append((time.perf_counter()-t0)/len(Xb)*1000)
    metrics['ms_per_sample'] = np.mean(times)

    all_metrics[name] = metrics
    all_preds[name] = (y_true, y_pred, y_prob)

In [ ]:
# Summary table
rows = []
for name, m in all_metrics.items():
    rows.append({
        'Model': DISPLAY[name],
        'Accuracy (%)': round(m['accuracy']*100, 2),
        'Macro F1': round(m['macro_f1'], 4),
        'Weighted F1': round(m['weighted_f1'], 4),
        'Macro AUC': round(m['macro_auc'], 4),
        'ms / sample': round(m['ms_per_sample'], 3),
    })

df = pd.DataFrame(rows).set_index('Model')
print('\n=== BENCHMARK RESULTS ===')
print(df.to_string())

In [ ]:
# 4 confusion matrices side-by-side
n_models = len(all_preds)
fig, axes = plt.subplots(1, n_models, figsize=(5*n_models, 5))
if n_models == 1: axes = [axes]

from sklearn.metrics import confusion_matrix
for ax, (name, (yt, yp, _)) in zip(axes, all_preds.items()):
    cm = confusion_matrix(yt, yp, normalize='true')
    sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=SHORT_NAMES, yticklabels=SHORT_NAMES,
                ax=ax, cbar=False, linewidths=0.4)
    ax.set_title(DISPLAY[name], fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')

plt.suptitle('Normalised Confusion Matrices', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/confusion_matrices_all.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ROC curves for best model (ResNet)
if 'resnet' in all_preds:
    yt, yp, yprob = all_preds['resnet']
    plot_roc_curves(yt, yprob,
                    save_path='../results/resnet_roc_curves.png',
                    title='ROC Curves — ResNet-1D (Best Model)')
    plot_pr_curves(yt, yprob,
                   save_path='../results/resnet_pr_curves.png',
                   title='PR Curves — ResNet-1D (Best Model)')

In [ ]:
# Per-class F1 comparison bar chart
plot_per_class_f1(all_metrics, save_path='../results/per_class_f1_comparison.png')
plt.show()

print('\nAll evaluation plots saved to ../results/')